In [14]:
import pandas as pd
from syslira_tools import PaperLibrary, ScopusClient, ZoteroClient, OpenAlexClient
from dotenv import load_dotenv
import os

In [15]:
# load open alex export

with open("../keep_snowballing.csv", "r") as f:
    oa_df = pd.read_csv(f)

library_df = oa_df

In [16]:
# remove duplicates
def find_duplicates_to_drop(papers_df) -> pd.DataFrame:
    # Find rows that have duplicate titles
    title_duplicates = papers_df[papers_df.duplicated(subset=["title"], keep=False)]

    if title_duplicates.empty:
        # No duplicates found, return empty dataframe
        return pd.DataFrame(columns=papers_df.columns)

    # handling logic for title duplicates
    grouped = title_duplicates.groupby('title')
    to_drop_df = pd.DataFrame(columns=papers_df.columns, dtype=str)

    for title, group in grouped:
        # prioritize journal articles over preprint and conference papers
        if any(group['type_crossref'] == 'journal-article'):
            to_keep = group[group['type_crossref'] == 'journal-article'].iloc[0]
        elif any(group['type_crossref'] == 'proceedings-article'):
            to_keep = group[group['type_crossref'] == 'proceedings-article'].iloc[0]
        else:
            to_keep = group.iloc[0]

        # Get all rows in this group EXCEPT the one we want to keep
        to_drop_from_group = group[group.index != to_keep.name]
        to_drop_df = pd.concat([to_drop_df, to_drop_from_group])

    return to_drop_df

# get and drop title duplicates
title_duplicates = find_duplicates_to_drop(library_df)
library_df_deduplicated = library_df.drop(title_duplicates.index)
# get and drop DOI duplicates (ignore empty DOIs)
doi_duplicates = library_df_deduplicated[library_df_deduplicated.duplicated(subset=["doi"], keep=False) & library_df_deduplicated["doi"].notna() & (library_df_deduplicated["doi"] != "")]
library_df_deduplicated = library_df_deduplicated.drop(doi_duplicates.index)

difference = len(library_df) - len(library_df_deduplicated)
print(f"Dropped {difference} duplicate papers")
library_df = library_df_deduplicated

Dropped 20 duplicate papers


/tmp/ipykernel_94299/1949532555.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  to_drop_df = pd.concat([to_drop_df, to_drop_from_group])


In [17]:
# drop anything before year 2022
library_df_year = library_df[library_df["publication_year"] >= 2022]
difference = len(library_df) - len(library_df_year)
print(f"Dropped {difference} papers before 2022")
library_df = library_df_year

Dropped 2 papers before 2022


In [18]:
# drop rows that are not a certain article type
valid_types = ['preprint', 'article', 'book-chapter', 'report', 'book', 'dissertation']
library_df_type = library_df[library_df["type"].isin(valid_types)]
difference = len(library_df) - len(library_df_type)
print(f"Dropped {difference} papers of invalid type")
library_df = library_df_type

Dropped 0 papers of invalid type


In [19]:
# remove retracted articles
library_df_no_retracted = library_df[library_df["is_retracted"] == False]
difference = len(library_df) - len(library_df_no_retracted)
print(f"Dropped {difference} retracted papers")
library_df = library_df_no_retracted

Dropped 0 retracted papers


In [20]:
# remove articles not in English
library_df_english = library_df[library_df["language"] == "en"]
difference = len(library_df) - len(library_df_english)
print(f"Dropped {difference} non-English papers")
library_df = library_df_english

Dropped 0 non-English papers


In [21]:
print(f"Final library has {len(library_df)} papers. Removed {len(oa_df) - len(library_df)} papers in total.")

Final library has 133 papers. Removed 22 papers in total.


In [22]:
# export to csv
library_df.to_csv("keep_snowballing_cleaned.csv", index=False)